# Norway Short Positions from the Finanstilsynet API

Qingyi (Freda) Song Drechsler

Norwegian Finanstilsynet presents the data in the json format through its API portal after the most recent website revamp, therefore the code does not resemble the other countries' download process.

In [1]:
# import packages
import json

import certifi
import pandas as pd
import urllib3

In [2]:
http = urllib3.PoolManager(cert_reqs='CERT_REQUIRED',
                           ca_certs=certifi.where())
url = "https://ssr.finanstilsynet.no/api/v2/instruments"
r = http.request('GET', url)
assert r.status == 200, f"Norway API returned status {r.status}"

data = json.loads(r.data.decode('utf-8'))
df = pd.json_normalize(data, 'events', ["isin", "issuerName"],
                       max_level=0)
dfValid = df.loc[df.shares != 0]
print(f"Norway: {df.shape[0]} raw records, "
      f"{dfValid.shape[0]} with a nonzero position")

Norway: 3364 raw records, 3187 with a nonzero position


In [3]:
df.head()

,date,shortPercent,shares,activePositions,isin,issuerName
0,2026-10-01T00:00:00,0.00,0,[],BMG9156K1018,2020 BULKERS
1,2026-09-30T00:00:00,1.80,413635,"[{'date': '2026-09-30T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS
2,2026-09-29T00:00:00,1.64,377008,"[{'date': '2026-09-29T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS
3,2026-09-25T00:00:00,1.43,329408,"[{'date': '2026-09-25T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS
4,2026-09-24T00:00:00,1.09,249963,"[{'date': '2026-09-24T00:00:00', 'shortPercent...",BMG9156K1018,2020 BULKERS


The field activePositions is a list of dictionary items that report detail information on short sellers and the respective holdings, and one needs to further open up this field to have a long table that shows each item as a separate row.

While most records show only one item in the activePositions field (i.e. one position holder's record), there are many cases where more than one position holder's records are presented in the activePositions field, organized in a dictionary. And note that the variable shortPercent actually represents the aggregated shortPercent reported by each position holder in that row.

For instance, below is the content of one activePositions:

[{'date': '2023-03-03',

'shortPercent': 0.68,

'shares': 153129,

'positionHolder': 'Voleon Capital Management LP'},

{'date': '2023-03-07',

'shortPercent': 0.89,

'shares': 198486,

'positionHolder': 'WorldQuant LLC'}]

As some of the records report a complete empty activePositions field, I first remove these records from the entire sample and then parse out the position holder information from this field, and then create a new dataframe that contains all the relevant fields.

In [4]:
# parse out the positionHolder info from the activePositions column
dateLst, holderLst, shortPctLst = [], [], []
shareLst, isinLst, issuerLst = [], [], []

for i in range(dfValid.shape[0]):
    thisIsin = dfValid.iloc[i].isin.__self__.to_dict()['isin']
    thisIssuer = dfValid.iloc[i].issuerName
    for pos in dfValid.iloc[i].activePositions:
        holderLst.append(pos['positionHolder'])
        shortPctLst.append(pos['shortPercent'])
        dateLst.append(pos['date'])
        shareLst.append(pos['shares'])
        isinLst.append(thisIsin)
        issuerLst.append(thisIssuer)

longDf = pd.DataFrame({
    'position_date': dateLst, 'isin': isinLst, 'issuer': issuerLst,
    'position_holder': holderLst, 'net_short_position': shortPctLst,
    'NumPositions': shareLst,
})

Last step is to remove the redundant rows from longDf: the reason is that the json data reports aggregate value by date per row, with each row possibly containing more than one position_holder.

When at least one of the position holders change position, it will lead to update in the aggregate short position, even though the rest of the position holders in that row might not report any changes. With that background in mind, when expanding the position_holders one by one, the ones that do not change positions will get repopulated day after day and therefore appear in duplicate records.

In [5]:
# the API reports one row per date with possibly several position
# holders, and holders who didn't change position get repeated on every
# subsequent date - drop those duplicates
longDf = longDf.drop_duplicates()
longDf.shape

(3537, 6)

And the final output for Norway looks like the following:

In [6]:
longDf.sort_values(['isin', 'position_holder', 'position_date']).head()

,position_date,isin,issuer,position_holder,net_short_position,NumPositions
800,2025-09-30T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.52,477063
799,2025-10-02T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.66,599968
798,2025-10-06T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.70,701581
797,2025-10-10T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.81,807536
796,2025-10-24T00:00:00,BMG0451H2087,ARCHER,WorldQuant LLC,0.78,783521
